# Free Market Data Provider Demo

This notebook runs the standalone `Provider` facade directly. It demonstrates daily data fetching, same-source adjustment factors, provider fallback, and field descriptions.

## 1. Import and Configure

The `Provider` facade receives concrete provider instances and an explicit provider order. Daily data is returned as raw OHLC plus `qfq_factor` and `hfq_factor`.

In [1]:
from __future__ import annotations

import os
import pandas as pd

from free_market_data.providers import DEFAULT_PROVIDER_CLASSES, Provider
from free_market_data.schema import DEFAULT_FIELD_DESCRIPTIONS

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

code = "600000.SH"
start_date = pd.Timestamp("2024-01-02")
end_date = pd.Timestamp("2024-01-10")
provider_order = ("baostock", "tencent", "akshare", "xueqiu", "yahoo")

provider_order

('baostock', 'tencent', 'akshare', 'xueqiu', 'yahoo')

## 2. Probe Each Provider

This cell tries each provider independently. A failure here usually means a network issue, API rate limit, missing package, or missing `XUEQIU_COOKIE`.

In [2]:
def probe_provider(name: str) -> dict:
    facade = Provider({name: DEFAULT_PROVIDER_CLASSES[name]()}, (name,))
    try:
        frame = facade.fetch_daily(code, start_date, end_date)
        return {
            "provider": name,
            "ok": True,
            "rows": len(frame),
            "price_sources": ",".join(sorted(frame.get("price_source", pd.Series(dtype=str)).dropna().astype(str).unique())),
            "columns": ", ".join(frame.columns),
            "error": "",
        }
    except Exception as exc:
        return {
            "provider": name,
            "ok": False,
            "rows": 0,
            "price_sources": "",
            "columns": "",
            "error": str(exc),
        }

provider_status = pd.DataFrame([probe_provider(name) for name in provider_order])
provider_status


1 Failed download:
['600000.SS']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


,provider,ok,rows,price_sources,columns,error
0,baostock,True,7,baostock,"date, stock_code, open, high, low, close, pre_...",
1,tencent,True,7,tencent,"date, stock_code, open, close, high, low, volu...",
2,akshare,False,0,,,"无法获取 600000.SH 的日线数据: {'akshare': ""('Connectio..."
3,xueqiu,False,0,,,"无法获取 600000.SH 的日线数据: {'xueqiu': ""雪球接口需要登录态。请设..."
4,yahoo,False,0,,,无法获取 600000.SH 的日线数据: {'yahoo': '返回空数据'}


## 3. Direct Daily Field Catalog

These are fields each provider declares as directly returned or directly mapped from its daily endpoint. Adjustment factors are generated by `Provider.fetch_daily`; other fields in this catalog are not post-computed by the database.

In [ ]:
schema = pd.DataFrame(DEFAULT_FIELD_DESCRIPTIONS)

catalog_rows = []
for provider_name in provider_order:
    provider_instance = DEFAULT_PROVIDER_CLASSES[provider_name]()
    for field in getattr(provider_instance, "daily_direct_fields", ()): 
        catalog_rows.append({"provider": provider_name, "field": field})

direct_daily_catalog = pd.DataFrame(catalog_rows)
direct_daily_catalog = direct_daily_catalog.merge(
    schema[["field", "description"]],
    on="field",
    how="left",
)
direct_daily_catalog

## 4. Fetch Daily Data Through the Final Provider

The final `Provider` traverses providers in order. A provider must return raw OHLC and same-source `qfq_factor`/`hfq_factor`; otherwise it is skipped for daily storage.

In [3]:
provider_map = {name: DEFAULT_PROVIDER_CLASSES[name]() for name in provider_order}
provider = Provider(provider_map=provider_map, provider_order=provider_order)

try:
    daily = provider.fetch_daily(code, start_date, end_date)
except Exception as exc:
    daily = pd.DataFrame()
    print(f"Provider.fetch_daily failed: {exc}")

daily


1 Failed download:
['600000.SS']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


,date,stock_code,open,high,low,close,pre_close,volume,amount,turnover,pct_change,pe_ttm,pb,ps_ttm,pcf_ttm,trade_status,is_st,change,amplitude,qfq_factor,hfq_factor,price_source,source,updated_at
0,2024-01-02,600000.SH,6.63,6.65,6.60,6.60,6.6200,22066700,146066303.7200,0.075200,-0.302100,5.006444,0.321265,1.089827,34.269302,1,0,-0.02,0.755287,0.936211,11.949786,baostock,baostock,2026-05-10 21:16:12
1,2024-01-03,600000.SH,6.59,6.65,6.59,6.64,6.6000,18203654,120639706.0100,0.062000,0.606100,5.036787,0.323213,1.096432,34.476995,1,0,0.04,0.909091,0.936211,11.949786,baostock,baostock,2026-05-10 21:16:12
2,2024-01-04,600000.SH,6.64,6.67,6.55,6.62,6.6400,28885978,190580609.9900,0.098400,-0.301200,5.021615,0.322239,1.093129,34.373149,1,0,-0.02,1.807229,0.936211,11.949786,baostock,baostock,2026-05-10 21:16:12
3,2024-01-05,600000.SH,6.60,6.76,6.59,6.68,6.6200,44421387,296976885.7900,0.151300,0.906300,5.067129,0.325160,1.103037,34.684688,1,0,0.06,2.567976,0.936211,11.949786,baostock,baostock,2026-05-10 21:16:12
4,2024-01-08,600000.SH,6.68,6.71,6.56,6.59,6.6800,37520337,247977824.9800,0.127800,-1.347300,4.998859,0.320779,1.088176,34.217379,1,0,-0.09,2.245509,0.936211,11.949786,baostock,baostock,2026-05-10 21:16:12
5,2024-01-09,600000.SH,6.60,6.64,6.54,6.61,6.5900,30741897,202647645.7300,0.104700,0.303500,5.014030,0.321752,1.091478,34.321226,1,0,0.02,1.517451,0.936211,11.949786,baostock,baostock,2026-05-10 21:16:12
6,2024-01-10,600000.SH,6.61,6.63,6.57,6.57,6.6100,22240946,146695926.3000,0.075800,-0.605100,4.983688,0.319805,1.084873,34.113533,1,0,-0.04,0.907716,0.936211,11.949786,baostock,baostock,2026-05-10 21:16:12


## 5. Check Same-Source Factors

`price_source` marks the provider that supplied the raw OHLC and both adjustment factors. `source` lists all providers that contributed fields to that row.

In [4]:
required_package = ["open", "high", "low", "close", "qfq_factor", "hfq_factor", "price_source"]

if daily.empty:
    print("No daily data was fetched. Check provider_status for errors.")
else:
    missing = [column for column in required_package if column not in daily.columns]
    assert not missing, f"Missing required daily package columns: {missing}"
    assert not daily[required_package].isna().any(axis=None), "Daily package contains missing raw prices or factors."

    factor_view = daily.assign(
        qfq_close=daily["close"] * daily["qfq_factor"],
        hfq_close=daily["close"] * daily["hfq_factor"],
    )
    display(daily[["date", "stock_code", "price_source", "source"]].drop_duplicates())
    display(factor_view[["date", "close", "qfq_factor", "qfq_close", "hfq_factor", "hfq_close"]].head())

,date,stock_code,price_source,source
0,2024-01-02,600000.SH,baostock,baostock
1,2024-01-03,600000.SH,baostock,baostock
2,2024-01-04,600000.SH,baostock,baostock
3,2024-01-05,600000.SH,baostock,baostock
4,2024-01-08,600000.SH,baostock,baostock
5,2024-01-09,600000.SH,baostock,baostock
6,2024-01-10,600000.SH,baostock,baostock


,date,close,qfq_factor,qfq_close,hfq_factor,hfq_close
0,2024-01-02,6.60,0.936211,6.178993,11.949786,78.868588
1,2024-01-03,6.64,0.936211,6.216441,11.949786,79.346579
2,2024-01-04,6.62,0.936211,6.197717,11.949786,79.107583
3,2024-01-05,6.68,0.936211,6.253889,11.949786,79.824570
4,2024-01-08,6.59,0.936211,6.169630,11.949786,78.749090


## 6. Available Fields and Descriptions

This view joins the actual columns returned by the provider with the local schema descriptions.

In [5]:
schema = pd.DataFrame(DEFAULT_FIELD_DESCRIPTIONS)

if daily.empty:
    fields = pd.DataFrame(columns=["field", "dtype", "description", "source"])
else:
    actual = pd.DataFrame({"field": list(daily.columns), "actual_dtype": [str(daily[column].dtype) for column in daily.columns]})
    fields = actual.merge(schema[["field", "dtype", "description", "source"]], on="field", how="left")

fields

,field,actual_dtype,dtype,description,source
0,date,datetime64[ns],datetime64[ns],交易日期,system
1,stock_code,object,object,标准化股票代码，如 600000.SH 或 000001.SZ,system
2,open,float64,float64,未复权开盘价,system
3,high,float64,float64,未复权最高价,system
4,low,float64,float64,未复权最低价,system
5,close,float64,float64,未复权收盘价,system
6,pre_close,object,float64,前收盘价,system
7,volume,object,float64,成交量,system
8,amount,object,float64,成交额,system
9,turnover,object,float64,换手率，通常为百分比数值,system


## 7. Optional: Request a Narrow Field Set

Even if you request a narrow field set, daily raw OHLC and the same-source adjustment factors stay protected because the database must always be able to save a complete daily package.

In [6]:
try:
    narrow = provider.fetch_daily(code, start_date, end_date, fields=["pe_ttm", "pb", "ps_ttm", "pcf_ttm", "trade_status", "is_st"])
except Exception as exc:
    narrow = pd.DataFrame()
    print(f"Narrow fetch failed: {exc}")

narrow.head()


1 Failed download:
['600000.SS']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


,date,stock_code,open,high,low,close,qfq_factor,hfq_factor,price_source,pe_ttm,pb,ps_ttm,pcf_ttm,trade_status,is_st,source,updated_at
0,2024-01-02,600000.SH,6.63,6.65,6.60,6.60,0.936211,11.949786,baostock,5.006444,0.321265,1.089827,34.269302,1,0,baostock,2026-05-10 21:17:44
1,2024-01-03,600000.SH,6.59,6.65,6.59,6.64,0.936211,11.949786,baostock,5.036787,0.323213,1.096432,34.476995,1,0,baostock,2026-05-10 21:17:44
2,2024-01-04,600000.SH,6.64,6.67,6.55,6.62,0.936211,11.949786,baostock,5.021615,0.322239,1.093129,34.373149,1,0,baostock,2026-05-10 21:17:44
3,2024-01-05,600000.SH,6.60,6.76,6.59,6.68,0.936211,11.949786,baostock,5.067129,0.325160,1.103037,34.684688,1,0,baostock,2026-05-10 21:17:44
4,2024-01-08,600000.SH,6.68,6.71,6.56,6.59,0.936211,11.949786,baostock,4.998859,0.320779,1.088176,34.217379,1,0,baostock,2026-05-10 21:17:44
